# JPEG compression feature extraction


In [9]:
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

from jpeg_compression_detection import (
    analyze_dct_spectrum,
    detect_jpeg_blockiness,
    detect_jpeg_compression,
    estimate_jpeg_quality,
)
from jpeg_compression_detection.paths import (
    DATA_DIR,
    GENERATED_DATA_DIR,
    display_path,
)

RAW_MULTIPLE_COMPRESSION_DIR = DATA_DIR / "multiple-compression-same-quality"
multiple_compression_dir = GENERATED_DATA_DIR / "multiple-compression-same-quality"
results_dir = GENERATED_DATA_DIR


# Generate images with repeated JPEG compression


In [ ]:
# This experiment creates many files. Enable it explicitly after reviewing the paths.
RUN_MULTIPLE_COMPRESSION_GENERATION = False
quality_levels = [10, 30, 50, 65, 75, 85, 95]


def _is_safe_generated_directory(path: Path) -> bool:
    """Return True only for a direct child of the configured generated-data directory."""
    return path.resolve().parent == GENERATED_DATA_DIR.resolve()


if not RUN_MULTIPLE_COMPRESSION_GENERATION:
    print("Generation skipped. Set RUN_MULTIPLE_COMPRESSION_GENERATION = True to run it.")
elif not RAW_MULTIPLE_COMPRESSION_DIR.is_dir():
    print(f"Generation skipped; raw input directory is missing: {display_path(RAW_MULTIPLE_COMPRESSION_DIR)}")
elif not _is_safe_generated_directory(multiple_compression_dir):
    raise RuntimeError(f"Refusing to modify a directory outside GENERATED_DATA_DIR: {display_path(multiple_compression_dir)}")
else:
    multiple_compression_dir.mkdir(parents=True, exist_ok=True)

    # Stage immutable PNG/PPM inputs in the generated-data workspace.
    for source_path in RAW_MULTIPLE_COMPRESSION_DIR.iterdir():
        if source_path.is_file() and source_path.suffix.lower() in {".png", ".ppm"}:
            shutil.copy2(source_path, multiple_compression_dir / source_path.name)

    # Remove derived files only from this explicitly validated generated-data directory.
    for path in multiple_compression_dir.iterdir():
        if path.is_file() and path.suffix.lower() not in {".png", ".ppm"}:
            path.unlink()
            print(f"Removed derived file: {path.name}")

    source_files = sorted(
        path for path in multiple_compression_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".png", ".ppm"}
    )
    for source_path in source_files:
        try:
            with Image.open(source_path) as image:
                rgb_image = image.convert("RGB")
                for quality in quality_levels:
                    output_path = multiple_compression_dir / f"{source_path.stem}_q{quality}.jpg"
                    rgb_image.save(output_path, "JPEG", quality=quality)
                    print(f"Saved {output_path.name}")
        except Exception as exc:
            print(f"Unable to convert {source_path.name}: {exc}")

    first_generation = sorted(multiple_compression_dir.glob("*.jpg"))
    for jpeg_path in first_generation:
        try:
            with Image.open(jpeg_path) as image:
                rgb_image = image.convert("RGB")
                for quality in quality_levels:
                    second_path = multiple_compression_dir / f"{jpeg_path.name}.q{quality}.jpg"
                    rgb_image.save(second_path, "JPEG", quality=quality)
                    print(f"Saved {second_path.name}")

                    with Image.open(second_path) as second_image:
                        for third_quality in quality_levels:
                            third_path = multiple_compression_dir / (
                                f"{jpeg_path.name}.q{quality}.q{third_quality}.jpg"
                            )
                            second_image.save(third_path, "JPEG", quality=third_quality)
                            print(f"Saved {third_path.name}")
        except Exception as exc:
            print(f"Unable to recompress {jpeg_path.name}: {exc}")


# Calculate compression indicators


In [11]:
image_files = (
    sorted(
        path
        for path in multiple_compression_dir.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".ppm"}
    )
    if multiple_compression_dir.is_dir()
    else []
)

if not image_files:
    print(f"No images to analyze in {display_path(multiple_compression_dir)}")

results = []
for image_path in image_files:
    print(f"\nAnalyzing: {image_path.name}")
    try:
        image = cv2.imread(str(image_path))
        if image is None:
            print("  Unable to load image")
            continue

        blockiness_ratios = detect_jpeg_blockiness(image)
        blockiness_ratio_max = np.max(blockiness_ratios)
        blockiness_ratio_mean = np.mean(blockiness_ratios)
        blockiness_ratio_std = np.std(blockiness_ratios)
        energy_mean, energy_std = analyze_dct_spectrum(image)
        estimated_quality, psnr_value = estimate_jpeg_quality(image)

        row = {
            "filename": image_path.name,
            "blockiness_ratio_max": blockiness_ratio_max,
            "blockiness_ratio_mean": blockiness_ratio_mean,
            "blockiness_ratio_std": blockiness_ratio_std,
            "energy_mean": energy_mean,
            "energy_std": energy_std,
            "estimated_quality": estimated_quality,
            "psnr": psnr_value,
        }
        for index, ratio in enumerate(blockiness_ratios):
            row[f"blockiness_ratio_{index}"] = ratio
        results.append(row)

        print(f"  Blockiness: max={blockiness_ratio_max:.3f}, mean={blockiness_ratio_mean:.3f}")
        print(f"  Energy: mean={energy_mean:.1f}, std={energy_std:.1f}")
        print(f"  Estimated quality: {estimated_quality}%, PSNR: {psnr_value:.2f} dB")
    except Exception as exc:
        print(f"  Error while processing {image_path.name}: {exc}")

print(f"\nProcessed {len(results)} images successfully")
df = pd.DataFrame(results)



Analyzing: 01 (1).png
  ✓ Blockiness: max=1.006, mean=1.000
  ✓ Energy: mean=201.2, std=268.8
  ✓ Estimated quality: 95%, PSNR: 45.13 dB

Analyzing: 01 (1).ppm
  ✓ Blockiness: max=1.006, mean=1.000
  ✓ Energy: mean=201.2, std=268.8
  ✓ Estimated quality: 95%, PSNR: 45.13 dB

Analyzing: 01 (1).ppm
  ✓ Blockiness: max=1.006, mean=1.000
  ✓ Energy: mean=201.2, std=268.8
  ✓ Estimated quality: 95%, PSNR: 45.13 dB

Analyzing: 01 (1)_q70.jpg
  ✓ Blockiness: max=1.006, mean=1.000
  ✓ Energy: mean=201.2, std=268.8
  ✓ Estimated quality: 95%, PSNR: 45.13 dB

Analyzing: 01 (1)_q70.jpg
  ✓ Blockiness: max=1.611, mean=1.007
  ✓ Energy: mean=115.1, std=252.7
  ✓ Estimated quality: 70%, PSNR: 74.20 dB

Analyzing: 01 (1)_q70.jpg.q70.jpg
  ✓ Blockiness: max=1.611, mean=1.007
  ✓ Energy: mean=115.1, std=252.7
  ✓ Estimated quality: 70%, PSNR: 74.20 dB

Analyzing: 01 (1)_q70.jpg.q70.jpg
  ✓ Blockiness: max=1.611, mean=1.007
  ✓ Energy: mean=115.1, std=252.6
  ✓ Estimated quality: 70%, PSNR: 75.12 dB

A

# Analyze and present the results


In [ ]:
if df.empty:
    print("Summary skipped because no image results are available.")
else:
    print("=" * 80)
    print("SUMMARY STATISTICS FOR ALL IMAGES")
    print("=" * 80)
    print(f"\nImages analyzed: {len(df)}")
    print("\n" + "-" * 80)
    print("BLOCKINESS RATIOS")
    print("-" * 80)
    print(df[["blockiness_ratio_max", "blockiness_ratio_mean", "blockiness_ratio_std"]].describe())

    print("\n" + "-" * 80)
    print("DCT ENERGY")
    print("-" * 80)
    print(df[["energy_mean", "energy_std"]].describe())

    print("\n" + "-" * 80)
    print("ESTIMATED QUALITY AND PSNR")
    print("-" * 80)
    print(df[["estimated_quality", "psnr"]].describe())

    print("\n" + "=" * 80)
    print("ESTIMATED-QUALITY DISTRIBUTION")
    print("=" * 80)
    for quality, count in df["estimated_quality"].value_counts().sort_index().items():
        bar = "█" * int(count / len(df) * 50)
        print(f"Quality {quality:3d}%: {count:3d} images {bar}")

    print("\n" + "=" * 80)
    print("FIVE IMAGES WITH THE HIGHEST BLOCKINESS")
    print("=" * 80)
    columns = ["filename", "blockiness_ratio_max", "estimated_quality", "psnr"]
    print(df.nlargest(5, "blockiness_ratio_max")[columns].to_string(index=False))

    print("\n" + "=" * 80)
    print("FIVE IMAGES WITH THE LOWEST BLOCKINESS")
    print("=" * 80)
    print(df.nsmallest(5, "blockiness_ratio_max")[columns].to_string(index=False))


## Visualizations


In [ ]:
if df.empty:
    print("Plots skipped because no image results are available.")
else:
    sns.set_style("whitegrid")
    plt.rcParams["figure.figsize"] = (16, 12)

    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle("JPEG compression-indicator analysis", fontsize=16, fontweight="bold")

    axes[0, 0].hist(df["blockiness_ratio_max"], bins=30, color="steelblue", edgecolor="black", alpha=0.7)
    axes[0, 0].set_title("Maximum blockiness distribution", fontweight="bold")
    axes[0, 0].set_xlabel("Maximum blockiness ratio")
    axes[0, 0].set_ylabel("Number of images")
    axes[0, 0].axvline(df["blockiness_ratio_max"].mean(), color="red", linestyle="--", label=f'Mean: {df["blockiness_ratio_max"].mean():.3f}')
    axes[0, 0].legend()

    axes[0, 1].hist(df["blockiness_ratio_mean"], bins=30, color="coral", edgecolor="black", alpha=0.7)
    axes[0, 1].set_title("Mean blockiness distribution", fontweight="bold")
    axes[0, 1].set_xlabel("Mean blockiness ratio")
    axes[0, 1].set_ylabel("Number of images")
    axes[0, 1].axvline(df["blockiness_ratio_mean"].mean(), color="red", linestyle="--", label=f'Mean: {df["blockiness_ratio_mean"].mean():.3f}')
    axes[0, 1].legend()

    axes[1, 0].hist(df["energy_mean"], bins=30, color="seagreen", edgecolor="black", alpha=0.7)
    axes[1, 0].set_title("Mean DCT-energy distribution", fontweight="bold")
    axes[1, 0].set_xlabel("Mean DCT energy")
    axes[1, 0].set_ylabel("Number of images")
    axes[1, 0].axvline(df["energy_mean"].mean(), color="red", linestyle="--", label=f'Mean: {df["energy_mean"].mean():.1f}')
    axes[1, 0].legend()

    axes[1, 1].hist(df["estimated_quality"], bins=20, color="orchid", edgecolor="black", alpha=0.7)
    axes[1, 1].set_title("Estimated JPEG-quality distribution", fontweight="bold")
    axes[1, 1].set_xlabel("Estimated quality (%)")
    axes[1, 1].set_ylabel("Number of images")
    axes[1, 1].axvline(df["estimated_quality"].mean(), color="red", linestyle="--", label=f'Mean: {df["estimated_quality"].mean():.1f}%')
    axes[1, 1].legend()

    first_scatter = axes[2, 0].scatter(df["blockiness_ratio_max"], df["estimated_quality"], c=df["psnr"], cmap="viridis", alpha=0.6, s=50)
    axes[2, 0].set_title("Blockiness vs estimated quality (color: PSNR)", fontweight="bold")
    axes[2, 0].set_xlabel("Maximum blockiness ratio")
    axes[2, 0].set_ylabel("Estimated quality (%)")
    plt.colorbar(first_scatter, ax=axes[2, 0], label="PSNR (dB)")

    second_scatter = axes[2, 1].scatter(df["energy_mean"], df["estimated_quality"], c=df["blockiness_ratio_max"], cmap="coolwarm", alpha=0.6, s=50)
    axes[2, 1].set_title("DCT energy vs estimated quality (color: blockiness)", fontweight="bold")
    axes[2, 1].set_xlabel("Mean DCT energy")
    axes[2, 1].set_ylabel("Estimated quality (%)")
    plt.colorbar(second_scatter, ax=axes[2, 1], label="Maximum blockiness ratio")

    plt.tight_layout()
    plt.show()


In [ ]:
if df.empty:
    print("Boundary plots skipped because no image results are available.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Blockiness-ratio distributions at each 8 x 8 block boundary", fontsize=14, fontweight="bold")

    for index in range(8):
        row, column = divmod(index, 4)
        axes[row, column].hist(df[f"blockiness_ratio_{index}"], bins=25, color=plt.cm.viridis(index / 8), edgecolor="black", alpha=0.7)
        axes[row, column].set_title(f"Boundary {index} (pixels {index}-{index + 1})", fontweight="bold")
        axes[row, column].set_xlabel("Blockiness ratio")
        axes[row, column].set_ylabel("Number of images")
        mean_value = df[f"blockiness_ratio_{index}"].mean()
        axes[row, column].axvline(mean_value, color="red", linestyle="--", label=f"mean={mean_value:.3f}")
        axes[row, column].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 80)
    print("MEAN BLOCKINESS AT EACH BOUNDARY")
    print("=" * 80)
    for index in range(8):
        mean_value = df[f"blockiness_ratio_{index}"].mean()
        std_value = df[f"blockiness_ratio_{index}"].std()
        print(f"Boundary {index} (pixels {index}-{index + 1}): mean={mean_value:.4f}, std={std_value:.4f}")

    boundary_7_mean = df["blockiness_ratio_7"].mean()
    other_boundaries_mean = df[[f"blockiness_ratio_{index}" for index in range(7)]].mean().mean()
    print("\n" + "=" * 80)
    print("BOUNDARY 7 COMPARED WITH THE OTHER BOUNDARIES")
    print("=" * 80)
    print(f"Mean blockiness at boundary 7: {boundary_7_mean:.4f}")
    print(f"Mean blockiness at other boundaries: {other_boundaries_mean:.4f}")
    print(f"Difference: {boundary_7_mean - other_boundaries_mean:.4f} ({(boundary_7_mean / other_boundaries_mean - 1) * 100:.1f}%)")


## Indicator correlation matrix


In [ ]:
if df.empty:
    print("Correlation analysis skipped because no image results are available.")
else:
    correlation_columns = [
        "blockiness_ratio_max",
        "blockiness_ratio_mean",
        "blockiness_ratio_std",
        "energy_mean",
        "energy_std",
        "estimated_quality",
        "psnr",
    ]
    correlation_matrix = df[correlation_columns].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt=".3f", cmap="coolwarm", center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title("Correlation between JPEG compression indicators", fontsize=14, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 80)
    print("STRONG CORRELATIONS (|r| > 0.5)")
    print("=" * 80)
    for first_index in range(len(correlation_columns)):
        for second_index in range(first_index + 1, len(correlation_columns)):
            value = correlation_matrix.iloc[first_index, second_index]
            if abs(value) > 0.5:
                print(f"{correlation_columns[first_index]:30s} <-> {correlation_columns[second_index]:30s}: r = {value:+.3f}")


## Export the results


In [ ]:
if df.empty:
    print("Export skipped because no image results are available.")
else:
    results_dir.mkdir(parents=True, exist_ok=True)
    full_results_path = results_dir / "analysis_results.csv"
    simplified_results_path = results_dir / "analysis_results_simplified.csv"
    summary_path = results_dir / "summary_stats.txt"

    df.to_csv(full_results_path, index=False, encoding="utf-8")
    simplified_columns = ["filename", "blockiness_ratio_max", "blockiness_ratio_mean", "energy_mean", "energy_std", "estimated_quality", "psnr"]
    df[simplified_columns].to_csv(simplified_results_path, index=False, encoding="utf-8")

    with summary_path.open("w", encoding="utf-8") as summary_file:
        summary_file.write("=" * 80 + "\n")
        summary_file.write("JPEG COMPRESSION ANALYSIS SUMMARY\n")
        summary_file.write("=" * 80 + "\n\n")
        summary_file.write(f"Images analyzed: {len(df)}\n")
        summary_file.write(f"Input directory: {display_path(multiple_compression_dir)}\n\n")
        summary_file.write("BLOCKINESS RATIOS\n" + "-" * 80 + "\n")
        summary_file.write(df[["blockiness_ratio_max", "blockiness_ratio_mean", "blockiness_ratio_std"]].describe().to_string())
        summary_file.write("\n\nDCT ENERGY\n" + "-" * 80 + "\n")
        summary_file.write(df[["energy_mean", "energy_std"]].describe().to_string())
        summary_file.write("\n\nESTIMATED QUALITY AND PSNR\n" + "-" * 80 + "\n")
        summary_file.write(df[["estimated_quality", "psnr"]].describe().to_string())
        summary_file.write("\n")

    print(f"Saved full results to: {display_path(full_results_path)}")
    print(f"Saved simplified results to: {display_path(simplified_results_path)}")
    print(f"Saved summary statistics to: {display_path(summary_path)}")
